In [2]:
import sys
print(sys.executable)

C:\Users\udome\Documents\naijacopybench\.venv\Scripts\python.exe


In [3]:
# Cell 0 — setup + freeze guard
import re, json, hashlib
from pathlib import Path
from collections import Counter

RAW = Path("../data/act/raw/CopyrightAct2022.txt")
EXPECTED_SHA = "11a195164e989fe6a3ea745dd4ba5b7615aee4050213257768432d4c2f39d52e"

text = RAW.read_text(encoding="utf-8")
actual = hashlib.sha256(text.encode("utf-8")).hexdigest()
assert actual == EXPECTED_SHA, f"HASH MISMATCH — not the frozen source!\n got {actual}"

lines = text.splitlines()
print(f"Freeze verified. {len(lines)} lines, {len(text):,} chars")

stats = {}   # accumulates everything → logs/eda_a.json

Freeze verified. 3039 lines, 153,016 chars


In [7]:
# Cell A1 — corpus scale
import math

n_chars = len(text)
n_words = len(text.split())
n_lines = len(lines)
est_tokens = n_chars // 4          # rough heuristic; ~4 chars/token

saullm_tokens = 30_000_000_000     # SaulLM-7B legal continued-pretraining corpus (post-cleaning)
ratio = saullm_tokens / est_tokens

stats["scale"] = {
    "chars": n_chars,
    "words": n_words,
    "lines": n_lines,
    "est_tokens": est_tokens,
    "saullm_tokens": saullm_tokens,
    "saullm_ratio": round(ratio, 0),                        # ~784,000x smaller
    "orders_of_magnitude_below": round(math.log10(ratio), 1),  # ~5.9
}

print(f"chars={n_chars:,}  words={n_words:,}  lines={n_lines:,}  ~tokens={est_tokens:,}")
print(f"SaulLM used 30B tokens after cleaning.")
print(f"This corpus is ~{stats['scale']['saullm_ratio']:,.0f}x smaller "
      f"(~{stats['scale']['orders_of_magnitude_below']} orders of magnitude below the floor).")

chars=153,016  words=26,097  lines=3,039  ~tokens=38,254
SaulLM used 30B tokens after cleaning.
This corpus is ~784,232x smaller (~5.9 orders of magnitude below the floor).


In [14]:
# Cell A2 — OCR pathology census
pathology = {}

# (a) non-ASCII characters — smart quotes, em-dashes, OCR garble
non_ascii = Counter(c for c in text if ord(c) > 127)
pathology["non_ascii_types"] = len(non_ascii)
pathology["non_ascii_top"] = non_ascii.most_common(15)

# (b) broken hyphenation across line breaks: word-\nword
broken_hyphen = re.findall(r"(\w+)-\n(\w+)", text)
pathology["broken_hyphenation"] = len(broken_hyphen)

# (c) repeated internal whitespace (OCR column artifacts)
multi_space = len(re.findall(r"\S  +\S", text))
pathology["multi_space_runs"] = multi_space

# (d) page furniture — running headers, BOTH directions + dotted TOC leaders
furniture_patterns = {
    "header_fwd":  r"Copyright Act, 2022\s+2023 No\. 8\s+A ?\d{2,3}",
    "header_rev":  r"A ?\d{2,3}\s+2023 No\. 8\s+Copyright Act, 2022",
    "toc_leader":  r"\.\.\s*\.\.\s*\.\.",     # dotted leaders in Arrangement
}
pathology["furniture"] = {k: len(re.findall(p, text)) for k, p in furniture_patterns.items()}

stats["ocr_pathology"] = pathology
print("non-ASCII char types:", pathology["non_ascii_types"])
for ch, n in pathology["non_ascii_top"]:
    print(f"   U+{ord(ch):04X} {ch!r}: {n}")
print("broken hyphenation:", pathology["broken_hyphenation"])
print("multi-space runs:", multi_space)
print("furniture:", pathology["furniture"])

# hunt for intra-word spurious spaces: lowercase fragment + space + lowercase fragment
# that look like one broken word. Heuristic: short lowercase token followed by
# a lowercase continuation that isn't a normal word.
import re
# crude but useful: find " ments", " tion", " ing" etc. as stranded suffixes
suspect_suffixes = re.findall(r"\b[a-z]{2,}\s(ment|ments|tion|tions|ing|ed|ance|ence)\b", text)
print("possible split-word suffixes:", len(suspect_suffixes))
print(Counter(suspect_suffixes).most_common(10))

# also: any lone 1-3 char lowercase line-internal fragments in the BODY (not margin)

non-ASCII char types: 5
   U+2014 '—': 215
   U+201C '“': 55
   U+201D '”': 55
   U+2019 '’': 37
   U+2013 '–': 6
broken hyphenation: 0
multi-space runs: 0
furniture: {'header_fwd': 34, 'header_rev': 34, 'toc_leader': 2}
possible split-word suffixes: 0
[]


In [10]:
# sanity-check the zeros: look for ANY hyphen-then-newline, and any 2+ space run
import re
print("any '-\\n' at all:", len(re.findall(r"-\n", text)))
print("any 3+ space run:", len(re.findall(r"   +", text)))
print("any 2+ space run:", len(re.findall(r"  +", text)))
# how is the text laid out? line-length profile tells you if it's wrapped or reflowed
lengths = [len(l) for l in lines if l.strip()]
import statistics as st
print("non-blank lines:", len(lengths))
print("mean line length:", round(st.mean(lengths), 1))
print("max line length:", max(lengths))
print("lines > 200 chars (reflowed paragraphs?):", sum(1 for x in lengths if x > 200))
print("lines < 40 chars (wrapped OR furniture?):", sum(1 for x in lengths if x < 40))
print()
# show the shape around s.19 so we know what A3 is walking through
for i in range(496, 512):
    print(f"L{i:4d} [{len(lines[i]):3d}]: {lines[i][:75]}")

any '-\n' at all: 0
any 3+ space run: 0
any 2+ space run: 0
non-blank lines: 2902
mean line length: 51.7
max line length: 1048
lines > 200 chars (reflowed paragraphs?): 1
lines < 40 chars (wrapped OR furniture?): 1048

L 496 [ 71]: 19.—(1) The duration of copyright conferred under this Act shall be, in
L 497 [ 13]: the case of —
L 498 [ 72]: (a) literary, musical or artistic works other than photographs, 70 years
L 499 [ 52]: after the end of the year in which the author dies ;
L 500 [ 72]: (b) works deriving copyrights from section 7 of this Act, 50 years after
L 501 [ 76]: the end of the year in which the work was first made available to the publi
L 502 [ 75]: or 50 years after the work was created, if not made available to the public
L 503 [ 18]: within that time ;
L 504 [ 12]: Broadcasting
L 505 [  8]: of works
L 506 [ 12]: incorporated
L 507 [ 24]: in an audiovisual work.
L 508 [  8]: Right to
L 509 [  8]: share in
L 510 [  8]: proceeds
L 511 [  8]: of sale.


In [11]:
# Cell A3 (corrected) — marginal-note furniture as CONTIGUOUS SHORT-LINE RUNS
# The margin column is OCR'd as a vertical stack of 1-3 word fragments.
# Detect RUNS of short lines, not single lines.

SHORT = 30   # chars; margin fragments are ~8-24 chars

def is_short(line):
    s = line.strip()
    return 0 < len(s) <= SHORT and not re.match(r"^\(?[a-z0-9ivx]+\)", s) \
           and not re.match(r"^\d{1,3}\.", s)

runs = []
i = 0
while i < len(lines):
    if is_short(lines[i]):
        j = i
        while j < len(lines) and is_short(lines[j]):
            j += 1
        if j - i >= 2:                      # a run of 2+ short lines = margin block
            block = " ".join(l.strip() for l in lines[i:j])
            runs.append({"start": i, "end": j-1, "n_lines": j-i, "text": block})
        i = j
    else:
        i += 1

stats["furniture_blocks"] = {
    "n_blocks": len(runs),
    "n_lines_total": sum(r["n_lines"] for r in runs),
}
print(f"furniture blocks (short-line runs >=2): {len(runs)}")
print(f"total lines consumed: {stats['furniture_blocks']['n_lines_total']}\n")
print("First 20 reconstructed blocks:")
for r in runs[:20]:
    print(f"  L{r['start']:4d}-{r['end']:<4d} ({r['n_lines']}): {r['text'][:60]}")

furniture blocks (short-line runs >=2): 53
total lines consumed: 444

First 20 reconstructed blocks:
  L   5-6    (2): Federal Republic of Nigeria Official Gazette
  L  26-29   (4): COPYRIGHT ACT, 2022 \ ARRANGEMENT OF SECTIONS Section :
  L 172-174  (3): Commencement. COPYRIGHT ACT, 2022 ACT No. 8
  L 209-213  (5): Objectives of the Act. Works eligible for copyright.
  L 250-270  (21): Ineligibility for copyright. Formalities. Copyright by virtu
  L 314-322  (9): Copyright by reference to international agreements. Nature o
  L 364-371  (8): Nature of copyright in audiovisual works. Nature of copyrigh
  L 413-418  (6): him as the author. Author’s moral rights. Nature of copyrigh
  L 459-464  (6): Right to remuneration for broadcasting of sound recordings.
  L 503-515  (13): within that time ; Broadcasting of works incorporated in an 
  L 558-559  (2): General exceptions.
  L 645-650  (6): instruction. Acts for purposes of instruction or examination
  L 692-713  (22): from another inst

In [12]:
# do the reconstructed blocks match known marginal titles?
# spot-check: does the s.19 region produce the expected two notes?
print("Blocks in s.16-19 region (L500-535):")
for r in runs:
    if 500 <= r["start"] <= 535:
        print(f"  L{r['start']}-{r['end']}: {r['text']}")

Blocks in s.16-19 region (L500-535):
  L503-515: within that time ; Broadcasting of works incorporated in an audiovisual work. Right to share in proceeds of sale. Commencements of rights. Duration of copyright.


In [16]:
# Arrangement of Sections titles → the authoritative furniture whitelist
toc_region = lines[26:172]          # adjust after eyeballing the real bounds
toc_titles = {}
for l in toc_region:
    m = re.match(r"^\s*(\d{1,3})\.\s+(.+?)\.?\s*$", l.strip())
    if m:
        num, title = int(m.group(1)), m.group(2).strip()
        toc_titles[num] = title

print(f"Arrangement titles extracted: {len(toc_titles)} (expect ~109)")
for n in sorted(toc_titles)[:8]:
    print(f"  s.{n}: {toc_titles[n]}")
print("  ...")
for n in sorted(toc_titles)[-4:]:
    print(f"  s.{n}: {toc_titles[n]}")
print(f"\nMissing from 1..109: {sorted(set(range(1,110)) - set(toc_titles))}")

print("s.16:", toc_titles[16])
print("s.17:", toc_titles[17])
print("s.18:", toc_titles[18])
print("s.19:", toc_titles[19])

Arrangement titles extracted: 109 (expect ~109)
  s.1: Objectives of the Act
  s.2: Works eligible for copyright
  s.3: Ineligibility for copyright
  s.4: Formalities
  s.5: Copyright by virtue of nationality or habitual residence
  s.6: Copyright by reference to country of origin
  s.7: Copyright in works of government, state authorities and international
  s.8: Copyright by reference to international agreements
  ...
  s.106: Repeals and savings provisions
  s.107: Transitional provisions
  s.108: Interpretation
  s.109: Citation

Missing from 1..109: []
s.16: Broadcasting of works incorporated in an audio-visual work
s.17: Right to share in proceeds of sale
s.18: Commencements of rights
s.19: Duration of copyright


In [17]:
import json
from pathlib import Path

Path("../data/derived").mkdir(parents=True, exist_ok=True)
Path("../data/derived/arrangement_titles.json").write_text(
    json.dumps(toc_titles, indent=2, ensure_ascii=False), encoding="utf-8"
)

stats["arrangement"] = {"n_titles": len(toc_titles), "missing": []}
print("saved arrangement_titles.json — 109 titles, complete")

saved arrangement_titles.json — 109 titles, complete


In [18]:
hdr = re.compile(r"^(\d{1,3})\.")
part1 = 175   # body starts after the Arrangement — adjust if needed
body_hits = [int(hdr.match(l).group(1)) for i, l in enumerate(lines)
             if i >= part1 and hdr.match(l)]
found = set(body_hits)
print(f"body section headers: {len(found)} unique, max={max(found)}")
print("missing 1..109:", sorted(set(range(1,110)) - found))
print("unexpected >109:", sorted(found - set(range(1,110))))
print("duplicates:", sorted({n for n in body_hits if body_hits.count(n) > 1}))

body section headers: 108 unique, max=108
missing 1..109: [109]
unexpected >109: []
duplicates: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [19]:
# where does the body actually begin? find first line that is a section header
# FOLLOWED by substantive provision text (not a bare TOC title)
for i, l in enumerate(lines):
    if re.match(r"^\d{1,3}\.[—\s]", l) and i > 150:
        # is the next non-blank line substantive (long) rather than another TOC title?
        nxt = lines[i+1].strip() if i+1 < len(lines) else ""
        print(f"L{i}: {l.strip()[:55]!r}  | next: {nxt[:40]!r}")
        if i > 185:
            break

L151: '95. Borrowing and investment power of the Commission.'  | next: 'PART XII — MISCELLANEOUS PROVISIONS'
L153: '96. Power to accept gifts.'  | next: '97. Regulations.'
L154: '97. Regulations.'  | next: '98. Application of the Pensions Reform A'
L155: '98. Application of the Pensions Reform Act.'  | next: '99. Power of the Minister to give direct'
L156: '99. Power of the Minister to give directives.'  | next: '100. Restriction on execution against pr'
L157: '100. Restriction on execution against property of the C'  | next: '101. Indemnity.'
L158: '101. Indemnity.'  | next: '102. Avoidance of conflict of interest.'
L159: '102. Avoidance of conflict of interest.'  | next: '103. Jurisdiction.'
L160: '103. Jurisdiction.'  | next: '104. Limitation on suit against the Comm'
L161: '104. Limitation on suit against the Commission.'  | next: '105. Service of documents.'
L162: '105. Service of documents.'  | next: '106. Repeals and savings provisions.'
L163: '106. Repeals and savings provision

In [23]:
hdr = re.compile(r"^(\d{1,3})\.")
part1 = 181                      # confirmed: body starts here
body_hits = [int(hdr.match(l).group(1)) for i, l in enumerate(lines)
             if i >= part1 and hdr.match(l)]
found = set(body_hits)
print(f"body section headers: {len(found)} unique, max={max(found)}")
print("missing 1..109:", sorted(set(range(1,110)) - found))
print("unexpected >109:", sorted(found - set(range(1,110))))
print("duplicates:", sorted({n for n in body_hits if body_hits.count(n) > 1}))

# and locate s.109 in the body directly, whatever its format
print("\ns.109 in body:")
for i, l in enumerate(lines):
    if i >= part1 and re.search(r"\b109\b", l):
        print(f"  L{i}: {l.strip()[:80]}")

body section headers: 108 unique, max=108
missing 1..109: [109]
unexpected >109: []
duplicates: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

s.109 in body:
  L2951: Citation. 109. This Act may be cited as the Copyright Act, 2022.


In [24]:
# how many body sections have a marginal note fused before the number?
# pattern: "Word(s). N. <text>"  where N is 1..109
prefixed = []
for i, l in enumerate(lines):
    if i >= 181:
        m = re.match(r"^([A-Z][a-z].*?\.)\s+(\d{1,3})\.\s", l)
        if m and 1 <= int(m.group(2)) <= 109:
            prefixed.append((i, int(m.group(2)), m.group(1)))
print(f"furniture-prefixed section headers: {len(prefixed)}")
for i, n, note in prefixed:
    print(f"  L{i}: s.{n} prefixed by {note!r}")

furniture-prefixed section headers: 1
  L2951: s.109 prefixed by 'Citation.'


In [25]:
for i, l in enumerate(lines):
    if i >= 181 and re.match(r"^(?:[1-9]|1[0-2])\.", l):
        print(f"L{i}: {l.strip()[:70]}")

L181: 1. The objectives of this Act are to —
L191: 2.—(1) Subject to this Act, the following works shall be eligible for
L226: 3. The following shall not be eligible for copyright —
L233: 4. Eligibility for copyright under this Act shall not require any form
L234: 5. Copyright shall be conferred by this section on every work that is
L240: 6. Copyright shall be conferred by this section on a work, which is a 
L247: 7. Copyright shall be conferred by this section on a work that is elig
L275: 8.—(1) Copyright shall be conferred by this section on a work, if —
L287: 9. Subject to the exceptions specified in Part II of this Act, copyrig
L306: 10.—(1) Subject to the exceptions specified in Part II of this Act,
L340: 11. Subject to the exceptions under Part II of this Act, copyright in 
L359: 12. Subject to the exceptions specified in Part II of this Act, copyri
L2959: 1. Subject to this Act and to section 27 of the Interpretation Act (wh
L2964: 2. Every meeting of the Board shall be presided

In [27]:
"""
Step 4 (Path B): Remove page furniture that is unambiguous and global.
  R1  running headers (68 lines, one per page)  -- exact pattern, zero risk
  R3  s.109 furniture-prefixed header           -- one known line
Marginal-note removal is DEFERRED to Step 5, done per-section with the
Arrangement whitelist, where section context makes it safe.

Reads:  data/act/raw/CopyrightAct2022.txt  (frozen; never edited)
Writes: data/act/clean/act_clean.txt
"""
import re, hashlib
from pathlib import Path

ROOT = Path.cwd().parent
RAW  = ROOT / "data/act/raw/CopyrightAct2022.txt"
OUT  = ROOT / "data/act/clean/act_clean.txt"
EXPECTED_SHA = "11a195164e989fe6a3ea745dd4ba5b7615aee4050213257768432d4c2f39d52e"

raw = RAW.read_text(encoding="utf-8")
sha = hashlib.sha256(raw.encode("utf-8")).hexdigest()
assert sha == EXPECTED_SHA, f"FREEZE VIOLATION: {sha} != {EXPECTED_SHA}"

lines = raw.split("\n")
n0 = len(lines)

# R1 — running headers, both directions (exact A2 patterns)
HEADER = re.compile(r"(Copyright Act, 2022\s+2023 No\. 8\s+A ?\d{2,3}"
                    r"|A ?\d{2,3}\s+2023 No\. 8\s+Copyright Act, 2022)")
kept = [l for l in lines if not HEADER.search(l)]
n_removed = n0 - len(kept)

text_out = "\n".join(kept)

# R3 — s.109 fused prefix
before = text_out.count("Citation. 109. This Act may be cited")
text_out = text_out.replace("Citation. 109. This Act may be cited",
                            "109. This Act may be cited")

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(text_out, encoding="utf-8")

# ---- verification gate ----
print(f"R1 removed {n_removed} header lines (expect ~68)")
print(f"R3 s.109 prefix fixed: {before} occurrence(s)")
d = len(raw) - len(text_out)
print(f"chars {len(raw):,} -> {len(text_out):,}  (removed {d:,}, {100*d/len(raw):.2f}%)")
for needle in ["70 years", "50 years"]:
    assert text_out.count(needle) > 0, f"LOST '{needle}'"
    print(f"'{needle}': {text_out.count(needle)} survive")
assert 0 < d < 6000, f"char delta {d} outside expected range for header-only strip"
print("OK — wrote", OUT)

R1 removed 68 header lines (expect ~68)
R3 s.109 prefix fixed: 1 occurrence(s)
chars 153,016 -> 149,478  (removed 3,538, 2.31%)
'70 years': 3 survive
'50 years': 8 survive
OK — wrote C:\Users\udome\Documents\naijacopybench\data\act\clean\act_clean.txt
